In [ ]:
%%sql -r dataframe_1
-- 1. Setup Environment
CREATE DATABASE IF NOT EXISTS LEARNCO;
CREATE SCHEMA IF NOT EXISTS LEARNCO.ANALYTICS;
USE SCHEMA LEARNCO.ANALYTICS;

-- 2. Create File Format for CSV
CREATE OR REPLACE FILE FORMAT csv_format
    TYPE = 'CSV'
    FIELD_DELIMITER = ','
    SKIP_HEADER = 1
    FIELD_OPTIONALLY_ENCLOSED_BY = '"'
    NULL_IF = ('NULL', '');

-- 3. Create Internal Stage (for manual uploads)
CREATE OR REPLACE STAGE learnco_stage
    FILE_FORMAT = csv_format;

# task 3.1 (a)

In [ ]:
USE SCHEMA LEARNCO.ANALYTICS;

-- 1. Students Table
CREATE OR REPLACE TABLE STUDENTS (
    student_id VARCHAR, first_name VARCHAR, last_name VARCHAR, email VARCHAR,
    age INT, country VARCHAR, registration_date DATE, subscription_type VARCHAR,
    total_courses_enrolled INT, is_active BOOLEAN, account_age_days INT
);

COPY INTO STUDENTS FROM @LEARNCO_STAGE/students.csv
FILE_FORMAT = (FORMAT_NAME = csv_format) ON_ERROR = 'CONTINUE';

-- 2. Courses Table
CREATE OR REPLACE TABLE COURSES (
    course_id VARCHAR, course_name VARCHAR, category VARCHAR, instructor_id VARCHAR,
    duration_hours INT, difficulty_level VARCHAR, price_usd FLOAT, is_free BOOLEAN,
    rating FLOAT, enrollment_count INT, published_date DATE, language VARCHAR,
    has_certificate BOOLEAN, revenue_per_enrollment FLOAT
);

COPY INTO COURSES FROM @LEARNCO_STAGE/courses.csv
FILE_FORMAT = (FORMAT_NAME = csv_format) ON_ERROR = 'CONTINUE';

-- 3. Assessments Table
CREATE OR REPLACE TABLE ASSESSMENTS (
    assessment_id VARCHAR, student_id VARCHAR, course_id VARCHAR, enrollment_date DATE,
    completion_date DATE, completion_status VARCHAR, score FLOAT, time_spent_hours FLOAT,
    quiz_attempts INT, certificate_issued BOOLEAN, feedback_rating INT, 
    last_activity_date DATE, completion_days INT, pass_flag BOOLEAN
);

COPY INTO ASSESSMENTS FROM @LEARNCO_STAGE/assessments.csv
FILE_FORMAT = (FORMAT_NAME = csv_format) ON_ERROR = 'CONTINUE';

-- 4. Course Performance Table
CREATE OR REPLACE TABLE COURSE_PERFORMANCE (
    course_id VARCHAR, course_name VARCHAR, category VARCHAR, total_enrollments INT,
    completion_rate FLOAT, avg_score FLOAT, avg_time_spent_hours FLOAT,
    avg_feedback_rating FLOAT, certificate_issued_count INT, drop_rate FLOAT
);

COPY INTO COURSE_PERFORMANCE FROM @LEARNCO_STAGE/course_performance.csv
FILE_FORMAT = (FORMAT_NAME = csv_format) ON_ERROR = 'CONTINUE';

-- 5. Student Engagement Table
CREATE OR REPLACE TABLE STUDENT_ENGAGEMENT (
    student_id VARCHAR, subscription_type VARCHAR, country VARCHAR, courses_enrolled INT,
    courses_completed INT, avg_score FLOAT, total_time_spent_hours FLOAT, certificates_earned INT
);

COPY INTO STUDENT_ENGAGEMENT FROM @LEARNCO_STAGE/student_engagenment.csv
FILE_FORMAT = (FORMAT_NAME = csv_format) ON_ERROR = 'CONTINUE';

# task 3.1(b)

In [ ]:
-- 1. Create the Masking Policy for Strings
CREATE OR REPLACE MASKING POLICY learner_pii_mask AS (val string) 
RETURNS string ->
  CASE
    WHEN CURRENT_ROLE() IN ('ADMIN', 'ACCOUNTADMIN') THEN val
    ELSE '##MASKED##'
  END;

-- 2. Apply the policy to the STUDENTS table columns
ALTER TABLE STUDENTS MODIFY COLUMN first_name SET MASKING POLICY learner_pii_mask;
ALTER TABLE STUDENTS MODIFY COLUMN last_name SET MASKING POLICY learner_pii_mask;
ALTER TABLE STUDENTS MODIFY COLUMN email SET MASKING POLICY learner_pii_mask;

# task 3.2
## a) Identifying "High Difficulty/Low Completion" Courses

In [ ]:
SELECT 
    course_id, 
    course_name, 
    completion_rate, 
    avg_score, 
    avg_time_spent_hours
FROM COURSE_PERFORMANCE
WHERE completion_rate < 40  -- Below 40%
  AND avg_score > 55          -- Above 55 average score
ORDER BY completion_rate ASC;

## b) Score Breakdown: Difficulty vs. Subscription

In [ ]:
SELECT 
    c.difficulty_level,
    s.subscription_type,
    ROUND(AVG(a.score), 2) AS avg_score
FROM ASSESSMENTS a
JOIN COURSES c ON a.course_id = c.course_id
JOIN STUDENTS s ON a.student_id = s.student_id
GROUP BY 1, 2
ORDER BY 1, 2;

## c) High-Churn Risk Students

In [ ]:
SELECT 
    student_id, 
    subscription_type,
    DATE_TRUNC('MONTH', enrollment_date) AS enrollment_month,
    COUNT(assessment_id) AS courses_enrolled,
    COUNT(CASE WHEN completion_status = 'Completed' THEN 1 END) AS courses_completed
FROM ASSESSMENTS a
JOIN STUDENTS s USING (student_id)
GROUP BY 1, 2, 3
HAVING courses_enrolled >= 3 
   AND (courses_completed / courses_enrolled) < 0.30
ORDER BY enrollment_month DESC;

## d) Instructor Ranking by Category

In [ ]:
WITH InstructorStats AS (
    SELECT 
        c.instructor_id,
        c.category,
        COUNT(c.course_id) AS course_count,
        AVG(cp.completion_rate) AS avg_completion_rate,
        AVG(cp.avg_feedback_rating) AS avg_feedback_rating
    FROM COURSES c
    JOIN COURSE_PERFORMANCE cp ON c.course_id = cp.course_id
    GROUP BY 1, 2
)
SELECT 
    instructor_id,
    category,
    course_count,
    avg_completion_rate,
    avg_feedback_rating,
    RANK() OVER (PARTITION BY category ORDER BY avg_completion_rate DESC) as category_rank
FROM InstructorStats
ORDER BY category, category_rank;